In [1]:
!pip install -qU datasets sentence-transformers

In [2]:
import pandas as pd
df = pd.read_csv('qa_dataset.csv', encoding='utf-8-sig')

entries = []

for index, row in df.iterrows():
    query = row["Позитивный вопрос"]
    positive = row["Абзац"]

    entrie = {
        "query": query,
        "positive": positive,
    }
    entries.append(entrie)

In [3]:
for entrie in entries[:3]:
    print(f"""query: {entrie['query']}\npositive: {entrie['positive'][:100]}...
=========================================================================\n\n""")

query: Какой уровень критичности дефекта может привести к неработоспособности приложений СУРЫ?
positive: 2.7.2 Терминология

Дефект -изъян или недочет в данных проекта. Любой дефект характеризуется обязате...


query: Как называется процедура выявления дефектов проекта с помощью тестов?
positive: 2.7.2 Терминология

Дефект -изъян или недочет в данных проекта. Любой дефект характеризуется обязате...


query: Что представляет собой уровень критичности 'Информация' в контексте дефектов проекта?
positive: 2.7.2 Терминология

Дефект -изъян или недочет в данных проекта. Любой дефект характеризуется обязате...




In [4]:
import random 
random.shuffle(entries)

# Делим на выборки
split_idx = int(0.8 * len(entries))

train_data = entries[:split_idx]
val_data = entries[split_idx:]

print(f"Train size: {len(train_data)}, Validation size: {len(val_data)}")

Train size: 1190, Validation size: 298


In [5]:
# Преобразуем в DataFrame и сохраняем
train_df = pd.DataFrame(train_data)
val_df = pd.DataFrame(val_data)

train_df.to_csv('train_data.csv', index=False, encoding='utf-8')
val_df.to_csv('val_data.csv', index=False, encoding='utf-8')

# Подготовка датасета в формате Hugging Face

In [6]:
import os
os.environ['WANDB_DISABLED'] = 'true'

In [7]:

from sentence_transformers import SentenceTransformer, InputExample, losses, evaluation
from torch.utils.data import DataLoader
from sentence_transformers import SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from sentence_transformers.evaluation import InformationRetrievalEvaluator
from datasets import Dataset
import torch

C:\Users\4789201\AppData\Local\Temp\ipykernel_15192\3837478435.py:1: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import SentenceTransformer, InputExample, losses, evaluation
C:\Users\4789201\AppData\Local\Temp\ipykernel_15192\3837478435.py:1: DeprecationWarning: Importing from 'sentence_transformers.evaluation' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.evaluation' instead.
  from sentence_transformers import SentenceTransformer, InputExample, losses, evaluation


In [8]:
def prepare_dataset(data):
    queries = []
    positives = []

    for item in data:
        queries.append(f"query: {item['query']}")
        positives.append(f"passage: {item['positive']}")

    return Dataset.from_dict({
        "anchor": queries,
        "positive": positives,
    })

train_dataset = prepare_dataset(train_data)
val_dataset = prepare_dataset(val_data)

In [9]:
# 3. Инициализация модели
model = SentenceTransformer("intfloat/multilingual-e5-base")

train_loss = losses.MultipleNegativesRankingLoss(model)

# 5. Подготовка Evaluator
def create_evaluator(dataset, train_data_1, test_data_1):
    import pandas as pd
    train_df = pd.DataFrame(train_data_1)
    test_df  = pd.DataFrame(test_data_1)

    # здесь вся выборка целиком
    whole_context = list(train_df.positive) + list(test_df.positive)

    corpus = {i: cont for i, cont in enumerate(whole_context)}

    test_df['idx'] = range(len(test_df))
    test_df = test_df.set_index('idx')

    queries = {}
    relevant_docs = {}

    for index, row in test_df.iterrows():
        q = row['query']
        queries[index] = q

        corrent_context_index = whole_context.index(row['positive'])
        relevant_docs[index] = [corrent_context_index]

    # print('queries', queries)

    # print('corpus', corpus)
    # print('relevant_docs', relevant_docs)

    return InformationRetrievalEvaluator(
        queries=queries,
        corpus=corpus,
        relevant_docs=relevant_docs,
        show_progress_bar=True,
        accuracy_at_k=[3, 5]
    )

evaluator = create_evaluator(val_dataset, train_data, val_data)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/multilingual-e5-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [10]:
from transformers import TrainerCallback

class MetricsLogger:
    def __init__(self):
        self.logs = []

    def log(self, metrics):
        self.logs.append(metrics)

    def save(self, path):
        pd.DataFrame(self.logs).to_csv(path, index=False)

class CustomLoggingCallback(TrainerCallback):
    def __init__(self, metrics_logger):
        self.metrics_logger = metrics_logger

    def on_evaluate(self, args, state, control, **kwargs):
        if state.log_history:
            self.metrics_logger.log(state.log_history[-1])

# Инициализация
metrics_logger = MetricsLogger()
callback = CustomLoggingCallback(metrics_logger)

In [12]:
# 6. Настройка аргументов обучения
training_args = SentenceTransformerTrainingArguments(
    output_dir="./e5-retriever",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    warmup_steps=100,
    learning_rate=4e-5,
    fp16=True,
    eval_strategy="steps",
    eval_steps=2,
    save_steps=2,
    save_total_limit=3,
    load_best_model_at_end=True,
)

# 7. Инициализация Trainer
trainer = SentenceTransformerTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    loss=train_loss,
    evaluator=evaluator,
)

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

In [ ]:
trainer.add_callback(callback)
trainer.train()
metrics_logger.save("./training_metrics.csv")

In [ ]:
model.save("/kaggle/working/e5_custom")

In [ ]:
results = evaluator(model)
print(evaluator.primary_metric)
print(results[evaluator.primary_metric])

In [ ]:
!zip -r /kaggle/working/e5_custom.zip /kaggle/working/e5_custom